# 02 — Classical Methods

Before deep RL, the field developed a rich set of tabular methods — algorithms that work when the state and action spaces are small enough to enumerate explicitly. These are the theoretical foundation; understanding them makes modern algorithms legible.

## Dynamic programming (narrative)

Dynamic programming (DP) methods — policy iteration and value iteration — solve MDPs exactly when the transition model P is known. They are the theoretical optimum: guaranteed convergence to V* and π*.

**Why they're not used in practice**: the transition model P(s'|s,a) is rarely known in advance and has to be learned or approximated. Also, DP requires storing V(s) for every state — intractable for large or continuous state spaces.

**Value iteration** is a simpler alternative to policy iteration: repeatedly apply the Bellman optimality operator until V converges. Faster per iteration but requires more iterations than policy iteration.

$$V_{k+1}(s) = \max_a \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma V_k(s')\right]$$

## Monte Carlo methods (narrative)

MC methods don't need P — they learn from complete episodes of experience. The value of a state is estimated as the average return observed across all episodes that visited that state.

**Why they're largely superseded**: MC methods require complete episodes before updating, which is slow and impossible in continuing (non-episodic) tasks. They also have high variance because the return G_t sums many uncertain future rewards.

MC methods remain useful for specific problems (e.g., evaluating a fixed policy in a game with clear episode boundaries) and as a conceptual reference. They appear in modern contexts as Monte Carlo Tree Search (MCTS), which powers AlphaGo and related systems — but MCTS is a planning algorithm, not a learning algorithm.

## Temporal Difference learning

TD learning combines the best of DP and MC:
- Like DP: updates happen at every step, not just at episode end
- Like MC: doesn't require a known model P — learns from experience

**TD(0) update rule** (the core of Q-learning and SARSA):
$$V(S_t) \leftarrow V(S_t) + \alpha\left[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)\right]$$

The quantity in brackets is the **TD error** δ: the difference between the current value estimate and the bootstrapped target R + γV(S'). The TD error is also the learning signal in actor-critic methods and a key quantity in neuroscience models of dopamine.

**Q-learning** extends this to action-values, with an off-policy (max) update:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\left[R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t)\right]$$

Tabular Q-learning is proven to converge to Q* given sufficient exploration and decreasing learning rates. It works well for small state spaces and is an excellent pedagogical tool.

In [ ]:
# Tabular Q-learning on GridWorld — making TD concrete
# Small enough to run to convergence; big enough to visualise learning dynamics

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

class GridWorldEnv:
    """Gym-compatible wrapper for GridWorld.""""
    def __init__(self, size=4):
        self.size = size
        self.terminals = {0, size*size - 1}
        self.reset()

    def reset(self):
        # Start in a random non-terminal state
        while True:
            self.state = np.random.randint(self.size * self.size)
            if self.state not in self.terminals:
                break
        return self.state

    def step(self, action):
        s = self.state
        if s in self.terminals:
            return s, 0.0, True, {}
        row, col = s // self.size, s % self.size
        dr, dc = [(-1,0),(1,0),(0,-1),(0,1)][action]
        nr = max(0, min(self.size-1, row + dr))
        nc = max(0, min(self.size-1, col + dc))
        ns = nr * self.size + nc
        self.state = ns
        done = ns in self.terminals
        return ns, -1.0, done, {}

def q_learning(env, n_episodes=5000, alpha=0.1, gamma=0.9, epsilon_start=1.0, epsilon_end=0.01):
    n_states = env.size * env.size
    n_actions = 4
    Q = np.zeros((n_states, n_actions))

    episode_returns = []
    td_errors = []

    for ep in range(n_episodes):
        epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * ep / n_episodes)
        s = env.reset()
        total_reward = 0
        ep_td_errors = []

        for _ in range(200):  # max steps per episode
            # ε-greedy action selection
            if np.random.rand() < epsilon:
                a = np.random.randint(n_actions)
            else:
                a = np.argmax(Q[s])

            ns, r, done, _ = env.step(a)
            total_reward += r

            # Q-learning update
            td_error = r + gamma * np.max(Q[ns]) - Q[s, a]
            Q[s, a] += alpha * td_error
            ep_td_errors.append(abs(td_error))

            s = ns
            if done:
                break

        episode_returns.append(total_reward)
        td_errors.append(np.mean(ep_td_errors))

    return Q, episode_returns, td_errors

env = GridWorldEnv(size=4)
Q, returns, td_errs = q_learning(env, n_episodes=5000)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Learning curve
window = 100
smoothed = np.convolve(returns, np.ones(window)/window, mode='valid')
axes[0].plot(smoothed, color='steelblue')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Return')
axes[0].set_title('Learning Curve (smoothed)', fontweight='bold')
axes[0].axhline(y=np.mean(returns[-500:]), color='red', linestyle='--', alpha=0.5, label=f'Final avg: {np.mean(returns[-500:]):.1f}')
axes[0].legend()

# TD error decay
td_smoothed = np.convolve(td_errs, np.ones(window)/window, mode='valid')
axes[1].plot(td_smoothed, color='darkorange')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Mean |TD Error|')
axes[1].set_title('TD Error Decay', fontweight='bold')

# Learned Q-value as V (max over actions)
V_learned = np.max(Q, axis=1).reshape(4, 4)
im = axes[2].imshow(V_learned, cmap='RdYlGn')
for i in range(4):
    for j in range(4):
        axes[2].text(j, i, f'{V_learned[i,j]:.1f}', ha='center', va='center', fontsize=10)
axes[2].set_title('Learned V(s) = max_a Q(s,a)', fontweight='bold')
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.savefig('/tmp/q_learning.png', dpi=100, bbox_inches='tight')
plt.show()


## Why tabular methods don't scale — and what comes next

Tabular Q-learning stores one Q-value per (state, action) pair. CartPole has a 4-dimensional continuous state space — infinitely many states. Atari has 210×160×3 pixel observations — 2^(~10^5) possible states.

The solution: **function approximation** — replace the Q-table with a neural network Q(s,a;θ). This is Deep Q-Networks (NB 03). The TD update still applies, but now it updates network weights θ instead of a table entry.

The same progression from tabular to function approximation applies to policy gradient methods, actor-critic methods, and everything else in modern deep RL.